# 🔬 Veritas Forensic — Academic Integrity & AI Detection Workbench (v1.2.0)
**Multi-User Collaborative Session via Google Colab + Cloudflare / LocalTunnel**

This notebook launches the complete **Veritas Forensic Workbench** in Google Colab so you and your fellow collaborator can simultaneously test documents, analyze OpenXML/PDF files, run ModernBERT & RoBERTa sequence classification, and view synchronized forensic reports in real time.

In [ ]:
# @title 1. Clone Veritas Repository and Install Node.js 22 & Dependencies
!git clone https://github.com/sparshdhand/veritas.git /content/veritas
%cd /content/veritas

# Setup Node.js 22 LTS in Colab environment
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
!sudo apt-get install -y nodejs

# Install project dependencies
!npm install --legacy-peer-deps
!node -v && npm -v

In [ ]:
# @title 2. Ensure Neural Models (ModernBERT + RoBERTa) & Build Webpack
%cd /content/veritas

# Ensure local model directories are created
!mkdir -p public/models/roberta-detector/onnx public/models/modernbert-detector/onnx

# Download ModernBERT ONNX weights if not bundled
!curl -f -L --retry 3 "https://huggingface.co/onnx-community/modernbert-ai-detection-raid-mage-ONNX/resolve/main/config.json" -o "public/models/modernbert-detector/config.json"
!curl -f -L --retry 3 "https://huggingface.co/onnx-community/modernbert-ai-detection-raid-mage-ONNX/resolve/main/tokenizer.json" -o "public/models/modernbert-detector/tokenizer.json"
!curl -f -L --retry 3 "https://huggingface.co/onnx-community/modernbert-ai-detection-raid-mage-ONNX/resolve/main/tokenizer_config.json" -o "public/models/modernbert-detector/tokenizer_config.json"
!curl -f -L --retry 3 "https://huggingface.co/onnx-community/modernbert-ai-detection-raid-mage-ONNX/resolve/main/special_tokens_map.json" -o "public/models/modernbert-detector/special_tokens_map.json"
!curl -f -L --retry 3 "https://huggingface.co/onnx-community/modernbert-ai-detection-raid-mage-ONNX/resolve/main/onnx/model_quantized.onnx" -o "public/models/modernbert-detector/onnx/model_quantized.onnx"

# Build frontend distribution
!npm run build

In [ ]:
# @title 3. Start Veritas Forensic Server & Generate Public Shareable URL
# Installs cloudflared to create an instant, secure public HTTPS tunnel with zero signup required
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!sudo dpkg -i cloudflared-linux-amd64.deb > /dev/null

import subprocess
import time
import re

%cd /content/veritas

# Start Node/Express fullstack server in background
server_proc = subprocess.Popen(["npx", "tsx", "server.ts"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(3)

# Launch cloudflared tunnel pointing to localhost:3000
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:3000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("Waiting for live tunnel URL...")
public_url = None
for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*60)
    print("🚀 VERITAS FORENSIC IS LIVE!")
    print(f"Share this URL with your collaborator:\n👉 {public_url}")
    print("="*60 + "\n")
else:
    print("Tunnel generation failed. Try running: !npx localtunnel --port 3000")
